# week12 · 슬라이드 병행 실습

**병행 슬라이드:** `CE100_W12b_ProgPrtc4.pptx` · 총 10장

원본의 기존 슬라이드 번호 유지, 마지막에 실습 연결 1쪽 추가. 출처 이력은 과목의 provenance 폴더 참고

**슬라이드 8번에서 연결된 원본 Colab**의 예제·변수명·파일명·자료·진행 순서를 바탕으로 실습. PPTX의 실제 슬라이드 순서를 기준으로 번호를 표시

슬라이드 개념 설명 후 해당 구간 실행. 구간별 **예측 → 원본 예제 → 한 조건 변경 → AI 질문·수정 지시 → 결과 검토** 진행. 차시별 AI 활동은 교수자가 선택

| 실습 구간 | 원본 주제 | PPTX 슬라이드 |
| --- | --- | --- |
| [S01](#s01) | 예제1 · 대기오염 기간 평균 | 3 |
| [S02](#s02) | 예제2 · Seoul·Busan 강수량 | 4 |
| [S03](#s03) | 예제3 · 지하수 수위 변화 | 5 |
| [S04](#s04) | 예제4 · 대기오염 시계열 | 6 |

Colab 연결 표시는 해당 슬라이드에서 여는 원본 노트북의 실습 구간을 의미. 세부 코드가 모두 슬라이드 한 장에 인쇄된 것은 아님. 선택 확장은 원본 예제를 유지하며 필요할 때 진행

## 실행 준비

다음 셀에서 실행 환경·자료 준비. 원본 변수명·파일명 유지. 기본 작업 위치는 임시 폴더. Drive 연결 실습 시 `USE_GOOGLE_DRIVE=True` 선택

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE100-engineering-programming/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

In [ ]:
# 수업 준비 셀: 아래 예제는 원본 변수명·파일명을 그대로 사용합니다.
from pathlib import Path
import os
import sys
import tempfile
import subprocess
import importlib.util

_in_colab = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
USE_GOOGLE_DRIVE = False  # 슬라이드의 Drive 연결 자체를 연습할 때 True로 변경
_lesson_dir = Path(tempfile.mkdtemp(prefix="ce100-slide-"))
_content_dir = Path("/content") if _in_colab else _lesson_dir
_wd_dir = _lesson_dir / "WD"
_wd_dir.mkdir(exist_ok=True)
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    _wd_dir = Path("/content/drive/MyDrive/WD")
    _wd_dir.mkdir(parents=True, exist_ok=True)
os.chdir(_lesson_dir)
print("수업 작업 폴더:", _lesson_dir)

# 저장소 data/DataW12.zip와 바이트가 같은 원본 ZIP 사본입니다.
# 비공개 저장소에서도 인증 없이 준비되도록 수업 노트북에 같은 사본을 포함합니다.
import base64
import hashlib
import io
import zipfile
_archive_bytes = base64.b64decode('UEsDBBQAAAAAALRVllkAAAAAAAAAAAAAAAAIACAARGF0YVcxMi91eAsAAQQAAAAABAAAAABVVA0AB0RvZ2dEb2dnRG9nZ1BLAwQUAAgACAAUZaZWAAAAAAAAAAAAAAAAGQAgAERhdGFXMTIvYWlyX3BvbGx1dGlvbi5jc3Z1eAsAAQQAAAAABAAAAABVVA0ABzjMVWRZ7adnRG9nZ2VUQU7EQAy7I/GTATVOOjO97xWJL3DgB/xfTALVxhmp2kO0dRzb9ePr57t9fsixfvB+vr7gAN4OWU/To0meoOnZZKaJNswmmibWVJr0NDkbRhNLk94UjDwaLsaZgZzfugJ5PCeLc+Eji7PSdkG9QjR2JWSx4JORz6bGyL3ykbEhz3q7XMEnIeOP85UmUm8HqmLQYJiQYRvyGdsTH2ycsXFefyiXLjJl13qlKKbC2xFpyV4g0pJ34U6LpYmxX/C0kBeItBScwXzgaSHlEWlhPp4WI2SRirzSUpClKI9IC98uRXlsacGWFkRajBledReOygeb8kC91NPCu1ZaCh9PCyuPXpWvaYGnhfho+J6V160l1H0n5TV8z8gaLVGQO6uh7jvdrrfvGefiSzV8513uuxFDbwneVVtCN981fOdLpW+7xsZnsvJ6+56QsXH+b4mE476z8u478/GWYOXd94LcN+RR1cCst+OqyntL8KVavi+LtGTlLdKSd1m0ROZjd0uMNCnK29YSFi0xGacob9ESIORnWn4BUEsHCDHNWCmUAQAAGQcAAFBLAwQUAAgACAAgSbhWAAAAAAAAAAAAAAAAFwAgAERhdGFXMTIvZ3JvdW5kd2F0ZXIuY3N2dXgLAAEEAAAAAAQAAAAAVVQNAAecVW1kWe2nZ0RvZ2dVlDtuxDAMRPs9i7MQh5Qs9ylziRTbpQxy/lAWsJwB3Az45xP9+f37Or5ef6+fBxrw0Sy/oz9bSRzxHCX9GGyNdJ4le0qUHCkp9lTrTHmWvLJQf0trKa+SlnW9JDKWrKsrKxlq7dKzDU11yoCmXdmVsjKjZWyNABNnQNqAZ2ayhmwDXa1DM5+aecpEuNI53tJbWmsiNyqEDTRKghaLBbRwYwEdbwpYQINTDbWuJklOegzYQEtao4mwgFbP2EDJ2aXJBBoc24kgNtAa4QZKsVPr8uqwgPKuEijHQncFl03eQCsz+PFjA7WSp4wA3pUvRtWVL0YV64tRsDPX9cWoXo4vRuLMR+ebUSt5SezNqGLNxFmOzhcjzmxBFHwxklRDU51E3xcjnleOzvfRlfN9dL0k1KpdIdSqu8KQTSYjHgFTnS8Z4T66atKNYmMfnZUEHU7so5slg5DFBuolB72N2ECp0KTfZijQ2EDb4x9QSwcIISLUz2sBAADnBQAAUEsDBBQACAAIADZJuFYAAAAAAAAAAAAAAAAbACAARGF0YVcxMi9yYWluZmFsbF9yZWdpb24uY3N2dXgLAAEEAAAAAAQAAAAAVVQNAAfIVW1kWe2nZ0RvZ2dtW0uOJbcR3Pss9KAy+d8aOoF8glmMDQEDaWHr/mZGsgW/iAJ6FWCzyGQwf8H3649///bH7+WX7//9UX79/tvv//r+8+ff/vnjjz9/Fn/c/v7EX7HxrTHqpT7fNqO1mCvairVvD6M95nVGR8zbGZ2lrW+T0fWK7tJkCfaUJbOaFdedmZfmsi6rxdu3xWiLGYzRs7Mt+7VRrOvXZowdjK6wmHxtx355F/6UpjO4HSsIeA7NZGFeS9OFeSut6lhsjRfmI2aQr52tmaLrzZC+S53ytXq2Nr5VRq3UTWbwoGmjk3CwtBFDHCzV/2+luqK9tIfI4EFSI4o5ODpoCx4crWwED476po15kPSc72L0bLfrWC8u0x6S2hQjHJKa7veQtOu3RnETex2OVrXt4Wgl1nhQ9Jwjm+ZQVC61B0UrXzQPjjrzzoOjVU3juH4ytsd+eROHo+yZPCgqZPSg6KJ11eCXM+sqCGY0QQ2CTTqxCoLxJalBMGeD1SDYYcLD6IwrKfMeLzjpICoYNmRhh2G62kOwM8Fk1IP5xujxFS8ztNgEL9f6K3rOwRWdETh4E4dij4LHCzaxQlCMb1QNijlTrAbFxF3VoJh3McOhWFWbH4q5WCG8IF/1Coopm/zFwdfwgof7PPZ4QdcDPl7Q+Ea0YCnHoxYkNfZXLUh6rNAZbRGVJ6Ndt9bSDcoEM8KcDF1BXVnYjtDFC7NHHWkLlp79yliPW8lfS5ZWRuHhZQZsjTd8WNrYvzY4Qt1xOELd26Gpsy9uQVN2Ii1Yao8s97B0ygoOSY1TgBYkPTZnix2SVr5pDSzlxK2BpS+rXTED2yZidSM+tmDp8fyf3O2Iyi+oa1zvQchjx8Ho2Vul9Xa4TWZDh9vkrKcHI/kC9iCksXl7EPKkNzzW4FsaoxYzPIy6ZnkdgVm3dgjZOAZ3ZI9DtnYIKdTrIOTLGsJtyiYOHxVNt2mMWrhCPp9wm+xGejByyLq8aWjvQUiJav26TbZYEJLTsY7kkZ1eD0IaB6UOQm4ibw+3eczweRLj1jjGqGvEHvCb7AtH0NR5FwM0fUGHJmQjaFqZOQOO0+koRvCUCTlA005GH4jufCdG0LRyijLgN/lmD+SPDx3bAE2nbOLQlEPVQHB/WdgKOwq6NWKPS1OeN6M7b+LQVI/HUZjyATt8y2S0x9jFaAQ7WcF8O8kI7uyOB4I71yIDJQ4Hmgm3yRFhgo9c707wkW/gDD5yMjODjpzozWAju7yJVJO3MIOMUh7MJCMPjSDObJ6oZTgkzSCjbtaqZgEzuLjoGGdQcen/D01vZlCxcXozg4rscSeYyCnaRLWt+3J4FZ42mMiZ6kSeyY54otxewgOH/+BNBBU5CZjBxcYhfCKEcztkgouysKAiX94ZDlP8x0I1xKRbSDQ5WV5BUL56C/6S3ccKglZO3BbCOi93BUUlWV6gKNcMC4kmW2yBo7rc4CjfsoW4zg5zoSs0iSMLiSa7oHXjuqxhFA6JCyx9mXapG11BU459CyxlNi0U3OxxFwpu5vlCU0iMEGGdi6wVJOUcZCGqs1dY8JdKpuAoR98VHOUccaEW4vCwXjm6g6OdmL+DonJT9+1bdkbRtxS0a+dzo2/JnmmDo2zbjZjOAXXfYqgRejjq7Nw2OpccovblqKD1eFcBW2E3uIOhJwGXoS8nuUFRTsA3SqGXj+3wV7wzx84ENY30G41L7iBvcJR7HBtNoSqnFsUQ59r7stQZnXp7NljKqfJGMaTzHppSRLMHSSZdlEBdOh+BIhPbjJ77pxP02O9kdBRqrgc4I5zIulbkd7KCLV2hg0YDacjYcJlDx7qYMVA00jujTQ44UIQ/Y3QInQJFkilj15sd4TN5uYePXVbg9vb/UQkpiH4Kmzzium7ssFFPJ8K669BzZrJZxz0TdMcxMHq4OGTWiqby5ylYMLSScwzUxTkGinbZwygixGa0x8IGoxBEFqPnHMk0hi46kyYuWhQxjdD0mLwEQ/uWtxYek40Drcce+VpqPTJD1+sLradWXUM0IgRckpoYpB6jlrtB6uFGvEHqeVlCcLQqWqVkstR69HyiXK9yPg6BTsZOKVjMbpdT1rs5LltKPXyn/BLSGFUZy6DqcFvKUtah/Mgg61DRZVB1uM9qkHWMvZhf6bEyGpXQJDBc5guKioH3YC69EIOswxHNIOtwMmaQdVztGDF8iBUMRR5bN9LMpevdQQae1x+lE4QddoTQdcQ/Q9dhhwNZR8gEWceXTjCkkWHQdcRl+a2F2Aq+RUU+aHSP+Po4ukeRfv7jz/98//1Ge7hNi5Mg1LPRSijcZmO0IYknsEc1JejI5RIKGdkYXXHs8jG4TR6bbpPnzWKINxxdd9OxuICDUbQCeQ0WNUNnEOrjw+hM50ToyrhK6I77w1sLtykTZKLJhxYsfXRsjQKUjzKqoaEf65nAEorygDccXtN0DSuVaELRv2U0aOqyhtCG6NA8y6HOYMrIBNasdQltZZMN/MqXxui9lIRGJvMwuBATCdxhbx4ajrQrigxtMeqRvcrYmm1eQlth2sKPNl2YQWCYjM7sdhOKqlaWsPPNySfqT2YyhBr6WgSiqcQfi/7mloVlXOf9IvnkdWVPic3o6EnL2BVV0+fHIJBXWm29uaMxWuMkZSwKt8VoD5crHxthXJlhxi17GF35CITQnbrfJ/r1BINQe5sBTzBkaC3sRaGPi8/+0scHoyPDHKEzi09CV2bQhO6Xdfkj9wzyOPIzQj1ClExQwzkL2vKVAKE99T1CkRQ7o5Hv85F9RWpCs2v5CYYHNLo6qY7zlYQ67hwJ2g3Uk1G0aRejaAnJ2A4NjMCowOVbMxNaQvGEQybYhZ0oxHF22NDGncNeu/W3oDXrekKvykpox/sLAkdUUjJ0FlcjGrqxstydFdon6mg/C4oEhGdwKCFsG/Qseb/RDhLTBkFf0AEhg8CZAg2hL+lHKuNMZijj3omLUMZroy1AGa9kRQjjjamYwvimhUEYf3TWUdjR9Ns/74yisyCL3RE4eVp7sgtBqOXLI0I93PhmtGpS0y8ZZd6etQahI+odGTuza0roSmWd0J0F3icaEZk5CmHcplgn2MgRFcI4h/T+xsZ+C235f9wztqNPTQ777aDLx/abxSreiwqKNyifzBv3+UZnFEnyZrQqR1IX53OHLs45DGRxZ4857tPfxejKYp9QOBZebuSNurWgqa4hdHGZNgqboRPgWaWM7YXvOlRxds+pinMoSVWc7xRUcWPfCFVcJ3DTRG681jVQxSt7sVTFt+whY7rMMGJe3tohKScrUMUl3RnXZfJ+65MvY/4fhSrufFNTFeeABlW8cgkBVdy4nIUsLmkjdPHKWS6EcYldEMaNjQNhXIgHZVz8DZRxZx8AZVzMC2kcD84IbRrT5m2hN0bz5R6Bs6gdI4KrFQzxgM8n25OVUcu2NqGeWhOhNeUFQvG2hk8tCSnz4nkp2waElIUt7VdAGpdrCW1cssHUxpmm0MbxPp1QZNXGaJXIDG1cmgXQxhsHlNTG2UOnNs5NCGjjUjRBG5ee1vpLdyTUtHpNbZzbGNDGTTd8WKrLzUqI7ZhP3mTaqXSCNu5c6UIb56QW2viLFdxSnSf0pRSCNi4xeN1imxeWsV2+Nopky1DHX4gTfpPLJqjjjZ30urFdUIS6zzVAHrcX1Au3JyGPSw6871NhGdvziR6h421avJ56GI2OkPz/zn77J5qtdR5rVvj2pDbOZwZtXC47xHHOoSGOQ64mdGgSDXGcfVtq49wqgzZeOYvf970br+tw9AX0fIdHaH3ZgjfNf/ftq/PG/L43JXRqpxvKuHNrMZVxAesjgeMq411Rl8NJZdyqoq1QkzmVceoypTCOR52EQuWRsSslP0J3Pr/6RIOLlEulMu4uOzP8AIG/FsXQ1rEa/K4yTs41lXHub6Qy3pp+beUvmwjdcpQpjbMSkdq4dbGOu5AsxXHu3aY47kvRns9YCB1SmF513HSGlY+wCdXWa8rj3CNNfVzO2JCA8hoMkZ06Y6mPGyV0qY87XaDUx40ykauPm469T04JXXIxUyB/WZlB2uIZzMSXpkAOWZbQmr+kIBTJtTEKFUu+huKWrZNvOGTeJV37VMgr34D8MaR+zTVPS4Vcl3uISj429XE9iHwmLP8/Xpc11e3lTyHVjL7zJy2faCSg7F78NpEWoy6lberjjbKL1Me5HXj18aZjR76HI3RKPpUCuVMNmAJ5q7Iyewp7jK+fPfLWzPVWpa5D6W4K5BzY/hLIeRNfj9wInW9GtyUZjn0JOzwWj4oGg3AjbBv3VL0JrSJTpEJeZWeRam6xow9pU1yBnH233zaSoDufUHyi9clfZBJqGbP/B1BLBwim5r/4wgwAAFg+AABQSwECFAMUAAAAAAC0VZZZAAAAAAAAAAAAAAAACAAYAAAAAAAAAAAA/0EAAAAARGF0YVcxMi91eAsAAQQAAAAABAAAAABVVAUAAURvZ2dQSwECFAMUAAgACAAUZaZWMc1YKZQBAAAZBwAAGQAYAAAAAAAAAAAAtoFGAAAARGF0YVcxMi9haXJfcG9sbHV0aW9uLmNzdnV4CwABBAAAAAAEAAAAAFVUBQABOMxVZFBLAQIUAxQACAAIACBJuFYhItTPawEAAOcFAAAXABgAAAAAAAAAAAC2gUECAABEYXRhVzEyL2dyb3VuZHdhdGVyLmNzdnV4CwABBAAAAAAEAAAAAFVUBQABnFVtZFBLAQIUAxQACAAIADZJuFam5r/4wgwAAFg+AAAbABgAAAAAAAAAAAC2gREEAABEYXRhVzEyL3JhaW5mYWxsX3JlZ2lvbi5jc3Z1eAsAAQQAAAAABAAAAABVVAUAAchVbWRQSwUGAAAAAAQABABrAQAAPBEAAAAA')
assert hashlib.sha256(_archive_bytes).hexdigest() == '9b5993cf69ea49b3da7fd71c26559bbc12bc3cc5affe820c8b501d7ec26ca1b5'
with zipfile.ZipFile(io.BytesIO(_archive_bytes)) as _bundle:
    for _entry in _bundle.infolist():
        if not _entry.is_dir():
            (_lesson_dir / Path(_entry.filename).name).write_bytes(_bundle.read(_entry))
print("원본 실습 데이터 준비 완료:", 'DataW12.zip')

## S01

**예제1 · 대기오염 기간 평균** · PPTX 3번

### 실행 전 예측

2022-01-01~2022-01-10 양 끝을 포함하면 관측은 몇 개인가? PM10·PM2.5 평균을 원자료로 확인

- 예상 결과와 이유:
- 실행 뒤 달라진 생각:

### 원본 예제

아래 예제의 변수·함수·자료를 슬라이드와 연결해서 읽기. 설치·경로·오류 처리 등 실행에 필요한 조정은 아래 변경 기록에 표시

## **예제1:** 대기 오염 데이터에서 특정 기간 동안 오염 농도의 평균 계산

**설명:**
1. `pandas` 라이브러리 설치
2. CSV 파일 읽기 (`air_pollution.csv`)
3. 특정 기간 동안 데이터 필터링
4. 오염 농도의 평균 계산
5. 결과 출력

### **Step 1**: `pandas` 라이브러리 설치
* 파이썬 환경에서 다음 명령어를 실행하여 pandas 라이브러리를 설치

**실행 조정 · 원본 셀 2:** 매번 설치하지 않고 pandas 확인. 핵심 예제의 입력·의미 유지

In [ ]:
import pandas as pd
print("pandas version:", pd.__version__)

### **Step 2**: CSV 파일 읽기 (`air_pollution.csv`)
* CSV 파일을 불러오기 위해 `pandas` 라이브러리 사용
* `pd.read_csv()` 함수를 이용하여 CSV 파일을 DataFrame으로 변환

In [ ]:
import pandas as pd

file_path = "air_pollution.csv"
df = pd.read_csv(file_path)

### **Step 3**: 특정 기간 동안 데이터 필터링
* DataFrame에서 특정 기간 동안의 데이터를 필터링하기 위해 불리언 인덱싱 사용
* 2022년 1월 1일부터 2022년 1월 10일까지의 데이터 필터링

In [ ]:
start_date = "2022-01-01"
end_date = "2022-01-10"
filtered_df = df[(df["Date"] >= start_date) & (df["Date"] <= end_date)]
print(filtered_df)

### **Step 4**: 오염 농도의 평균 계산
* 필터링된 데이터에서 오염 농도의 평균을 구하기 위해 `mean()` 함수 사용
* 이 예제에서는 PM10과 PM2.5의 평균 계산



In [ ]:
pm10_average = filtered_df["PM10"].mean()
pm25_average = filtered_df["PM2.5"].mean()

### **Step 5-1**: 평균값 출력

In [ ]:
unit_pm = "\u03BCg/m\u00B3"

print(f"PM10 평균 농도: {pm10_average:.2f} {unit_pm}")
print(f"PM2.5 평균 농도: {pm25_average:.2f} {unit_pm}")

### **Step 5-2**: Box plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(data=filtered_df[['PM10', 'PM2.5']])
plt.ylabel(f'Concentration ({unit_pm})')
plt.title('Concentration of PM10 and PM2.5')
plt.show()

In [ ]:
assert len(filtered_df) == 10
assert pm10_average == sum(filtered_df["PM10"])/len(filtered_df)
print("원본 예제 확인 완료")

### 이해 확인·변형

기간을 하루로 바꾸고 그날 원자료와 평균이 같은지 확인

- 바꾼 한 가지 조건:
- 예상 / 실제 결과 / 이유:

### AI에게 질문하고 검토하기

Date·PM10·PM2.5, 기간 양 끝 포함, 결측 정책, 농도 단위를 명시하고 선택된 행을 먼저 확인 요청

원본 예제의 이름·입력을 사용해 **목적·입력·출력·조건·기대값** 전달. AI에게 모호한 조건·처리 계획부터 설명 요청. AI 사용이 어려우면 동료와 같은 요청문 검토

- 내가 보낸 질문과 첫 답변의 부족한 점:
- 새 반례(입력·기대 결과·기대값의 근거):
- 수정 지시와 최종 결과를 채택/거절한 이유:

## S02

**예제2 · Seoul·Busan 강수량** · PPTX 4번

### 실행 전 예측

Seoul과 Busan의 관측 수·평균·최댓값을 따로 구해야 하는 이유는?

- 예상 결과와 이유:
- 실행 뒤 달라진 생각:

### 원본 예제

아래 예제의 변수·함수·자료를 슬라이드와 연결해서 읽기. 설치·경로·오류 처리 등 실행에 필요한 조정은 아래 변경 기록에 표시

## **예제2:** 강수량 데이터를 사용하여 특정 지역의 평균 및 최대 강우량 계산

**설명:**
1. CSV 파일 읽기 (`rainfall_region.csv`)
2. 특정 지역의 데이터 필터링
3. 평균 및 최대 강우량 계산
4. 결과 출력

### **Step 1**: CSV 파일 읽기 (`rainfall_region.csv`)
* CSV 파일을 불러오기 위해 `pandas` 라이브러리 사용
* `pd.read_csv()` 함수를 이용하여 CSV 파일을 DataFrame으로 변환

In [ ]:
import pandas as pd

file_path = "rainfall_region.csv"
df = pd.read_csv(file_path)

### **Step 2**: 특정 지역의 데이터 필터링
* DataFrame에서 특정 지역의 데이터를 필터링하기 위해 불리언 인덱싱 사용
* 이 예제에서는 서울 (Seoul) 지역의 데이터 필터링

In [ ]:
df_seoul = df[df["Region"] == "Seoul"]
df_busan = df[df["Region"] == "Busan"]

### **Step 3**: 평균 및 최대 강우량 계산
* 필터링된 데이터에서 평균 강수량과 최대 강수량을 구하기 위해 `mean()` 함수와 `max()` 함수 사용

In [ ]:
average_seoul = df_seoul["Rainfall"].mean()
max_seoul = df_seoul["Rainfall"].max()

average_busan = df_busan["Rainfall"].mean()
max_busan = df_busan["Rainfall"].max()

### **Step 4-1**: 평균 및 최대값값 출력

In [ ]:
print(f"서울 지역의 평균 강우량: {average_seoul:.2f} mm")
print(f"서울 지역의 최대 강우량: {max_seoul:.2f} mm")

print(f"부산 지역의 평균 강우량: {average_busan:.2f} mm")
print(f"부산 지역의 최대 강우량: {max_busan:.2f} mm")

### **Step 4-2**: Box plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(x=['Seoul']*len(df_seoul) + ['Busan']*len(df_busan), y=df_seoul['Rainfall'].tolist() + df_busan['Rainfall'].tolist())
plt.ylabel('Rainfall (mm)')
plt.title('Rainfall amounts of Seoul and Busan')
plt.show()

In [ ]:
assert set(df_seoul["Region"]) == {"Seoul"}
assert set(df_busan["Region"]) == {"Busan"}
assert max_seoul == df_seoul["Rainfall"].max()
print("원본 예제 확인 완료")

### 이해 확인·변형

지역 조건을 빠뜨린 사본을 만들고 두 지역의 결과가 섞이는지 확인

- 바꾼 한 가지 조건:
- 예상 / 실제 결과 / 이유:

### AI에게 질문하고 검토하기

Region·Rainfall 열, 지역명, 단위 mm와 각 지역의 유효 관측 수를 포함해 요청

원본 예제의 이름·입력을 사용해 **목적·입력·출력·조건·기대값** 전달. AI에게 모호한 조건·처리 계획부터 설명 요청. AI 사용이 어려우면 동료와 같은 요청문 검토

- 내가 보낸 질문과 첫 답변의 부족한 점:
- 새 반례(입력·기대 결과·기대값의 근거):
- 수정 지시와 최종 결과를 채택/거절한 이유:

## S03

**예제3 · 지하수 수위 변화** · PPTX 5번

### 실행 전 예측

Level의 평균·표준편차와 날짜 축 확인. 문자열과 datetime 날짜는 같은가?

- 예상 결과와 이유:
- 실행 뒤 달라진 생각:

### 원본 예제

아래 예제의 변수·함수·자료를 슬라이드와 연결해서 읽기. 설치·경로·오류 처리 등 실행에 필요한 조정은 아래 변경 기록에 표시

## **예제3:** 지하수 수위 변화 분석

**설명:**
1. CSV 파일 읽기 (`groundwater.csv`)
2. 데이터를 읽어 지하수 수위의 평균 및 표준편차 계산
3. 시간에 따른 지하수 수위의 변화를 선 그래프로 시각화

### Data analysis

**실행 조정 · 원본 셀 27:** 원본 Date를 실제 datetime으로 해석하여 주 단위 날짜 축과 일치시킴. 핵심 예제의 입력·의미 유지

In [ ]:
# Importing necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Read data
data = pd.read_csv('groundwater.csv', parse_dates=['Date']).sort_values('Date')

# Calculate average and std of groundwater level
average_level = data['Level'].mean()
print(f"Average: {average_level:.2f} m")

std_level = data['Level'].std()
print(f"Std: {std_level:.2f} m")

# Visualization
fig, ax = plt.subplots()
ax.plot(data['Date'], data['Level'], label='Groundwater Level')
ax.axhline(y=average_level, color='r', linestyle='--', label='Mean')
ax.fill_between(data['Date'], average_level - std_level, average_level + std_level, color='gray', alpha=0.2, label='±1 std')

ax.set_xlabel('Date (yyyy-mm-dd)')
ax.set_ylabel('Level (m)')
ax.set_title('Groundwater Level Change')

# Set x-tick labels to appear weekly
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45)
plt.show()

In [ ]:
assert data["Date"].is_monotonic_increasing
assert abs(average_level-data["Level"].mean()) < 1e-12
print("원본 예제 확인 완료")

### 이해 확인·변형

시간 순서와 주 단위 눈금이 실제 Date와 맞는지 확인

- 바꾼 한 가지 조건:
- 예상 / 실제 결과 / 이유:

### AI에게 질문하고 검토하기

수위 단위·날짜 해석·정렬·표준편차 정의를 명시하고 그림의 좌표도 검토 요청

원본 예제의 이름·입력을 사용해 **목적·입력·출력·조건·기대값** 전달. AI에게 모호한 조건·처리 계획부터 설명 요청. AI 사용이 어려우면 동료와 같은 요청문 검토

- 내가 보낸 질문과 첫 답변의 부족한 점:
- 새 반례(입력·기대 결과·기대값의 근거):
- 수정 지시와 최종 결과를 채택/거절한 이유:

## S04

**예제4 · 대기오염 시계열** · PPTX 6번

### 실행 전 예측

기간 평균과 전체 기간 시계열의 평균이 같은 통계인지 구분

- 예상 결과와 이유:
- 실행 뒤 달라진 생각:

### 원본 예제

아래 예제의 변수·함수·자료를 슬라이드와 연결해서 읽기. 설치·경로·오류 처리 등 실행에 필요한 조정은 아래 변경 기록에 표시

## **예제4:** 대기 오염 데이터 분석

**설명:**
1. CSV 파일 읽기 (`air_pollution.csv`)
2. 데이터를 읽어 PM2.5 및 PM10의 평균 및 표준편차 계산
3. 시간에 따른 오염물질 농도 변화를 선 그래프로 시각화

**실행 조정 · 원본 셀 29:** 원본 Date를 실제 datetime으로 해석하여 주 단위 날짜 축과 일치시킴. 핵심 예제의 입력·의미 유지

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# Read data
data = pd.read_csv('air_pollution.csv', parse_dates=['Date']).sort_values('Date')

# Calculate average PM2.5 and PM10 concentration
average_pm25 = data['PM2.5'].mean()
average_pm10 = data['PM10'].mean()
std_pm25 = data['PM2.5'].std()
std_pm10 = data['PM10'].std()

unit_pm = "\u03BCg/m\u00B3"
print(f"Average PM2.5 concentration: {average_pm25:.2f} {unit_pm}")
print(f"Average PM10 concentration: {average_pm10:.2f} {unit_pm}")

# Visualization
fig, ax = plt.subplots()
ax.plot(data['Date'], data['PM2.5'], label='PM2.5')
ax.plot(data['Date'], data['PM10'], label='PM10')

ax.axhline(y=average_pm25, color='b', linestyle='--', label='Mean PM2.5')
ax.axhline(y=average_pm10, color='orange', linestyle='--', label='Mean PM10')

ax.fill_between(data['Date'], average_pm25 - std_pm25, average_pm25 + std_pm25, color='blue', alpha=0.2, label='PM2.5 ±1 std')
ax.fill_between(data['Date'], average_pm10 - std_pm10, average_pm10 + std_pm10, color='orange', alpha=0.2, label='PM10 ±1 std')

ax.set_xlabel('Date')
ax.set_ylabel(f'Concentration ({unit_pm})')
ax.set_title('Air Pollution Concentration Change')

# Set x-tick labels to appear weekly
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45)

plt.legend()
plt.show()

In [ ]:
assert data["Date"].is_monotonic_increasing
assert average_pm10 == data["PM10"].mean()
print("원본 예제 확인 완료")

### 이해 확인·변형

1월 1~10일과 전체 기간을 각각 계산해 포함된 관측 수·기간 함께 기록

- 바꾼 한 가지 조건:
- 예상 / 실제 결과 / 이유:

### AI에게 질문하고 검토하기

시계열·평균·표준편차의 기간과 단위, 자료에서 말할 수 있는 결론의 범위 명시

원본 예제의 이름·입력을 사용해 **목적·입력·출력·조건·기대값** 전달. AI에게 모호한 조건·처리 계획부터 설명 요청. AI 사용이 어려우면 동료와 같은 요청문 검토

- 내가 보낸 질문과 첫 답변의 부족한 점:
- 새 반례(입력·기대 결과·기대값의 근거):
- 수정 지시와 최종 결과를 채택/거절한 이유:

## 원본 예제로 기능 완성하기

원본 예제1을 pollution_mean(frame,start,end)로 작성. Date(ISO 날짜 문자열), PM10, PM2.5 표에서 양 끝 포함 기간의 각 오염물질 평균을 같은 이름의 딕셔너리로 반환. 각 열의 결측은 따로 제외하며 유효값이 없으면 None. start>end는 ValueError, 원본은 보존

코드 작성 전 입력·출력 예와 처리 단계를 자신의 말로 기록. AI의 힌트·구현 활용 가능. 시험 기대값은 직접 확인

- 입력 / 기대 출력 / 처리 단계:

In [ ]:
def pollution_mean(frame,start,end):
    raise NotImplementedError("TODO: 원본 예제1의 기간·유효 관측·농도")

### 공개 점검

시험 입력·기대값 확인 후 실행. **“미완료”는 통과가 아님.** TODO 완성 후 재실행, 직접 만든 반례 추가

In [ ]:
def check_answer(candidate):
    frame=pd.read_csv("air_pollution.csv")
    before=frame.copy(deep=True)
    result=candidate(frame,"2022-01-01","2022-01-10")
    selected=frame.loc[frame["Date"].between("2022-01-01","2022-01-10")]
    for name in ["PM10","PM2.5"]:
        values=selected[name].dropna().tolist()
        assert abs(result[name]-sum(values)/len(values))<1e-10
    small=pd.DataFrame({"Date":["2022-01-01","2022-01-02"],"PM10":[0.,10.],"PM2.5":[None,4.]})
    assert candidate(small,"2022-01-01","2022-01-02")=={"PM10":5.,"PM2.5":4.}
    assert candidate(small,"2030-01-01","2030-01-02")=={"PM10":None,"PM2.5":None}
    try:
        candidate(frame,"2022-01-10","2022-01-01")
    except ValueError:
        pass
    else:
        raise AssertionError("역전된 기간을 거절해야 합니다")
    pd.testing.assert_frame_equal(frame,before)
    print("공개 점검 통과: 원본 데이터·기간·결측·빈 선택·보존")

try:
    check_answer(pollution_mean)
except NotImplementedError:
    print("미완료: TODO를 완성한 뒤 공개 점검을 다시 실행하세요.")

## 마무리 기록

- 슬라이드의 핵심 원리를 내 말로 설명:
- AI에게 전달한 입력·출력·실패 조건:
- 실행 성공 외에 결과가 맞다는 근거:
- 새로 시험한 반례와 수정 지시:
- 다른 입력·환경에 적용할 때 추가로 확인할 조건:

[학생 학습 가이드](../LEARNING_GUIDE.md) · [슬라이드 대응표](../SLIDE_ALIGNMENT.md)

## 원본과의 실행 조정 기록

- 원본 셀 2: 매번 설치하지 않고 pandas 확인
- 원본 셀 27: 원본 Date를 실제 datetime으로 해석하여 주 단위 날짜 축과 일치시킴
- 원본 셀 29: 원본 Date를 실제 datetime으로 해석하여 주 단위 날짜 축과 일치시킴